# 02. LangChain Basic v1.x

## 학습 목표

- ChatOpenAI 모델을 호출한다.
- `ChatPromptTemplate`으로 프롬프트를 구성한다.
- LCEL 방식으로 `prompt | model | parser` 체인을 만든다.

## 공통 전제

- 실습 데이터: `../data/공직자_민원응대_핵심_매뉴얼.pdf`
- 기본 모델: `gpt-4o-mini`
- 기본 임베딩 모델: `text-embedding-3-small`
- 벡터DB: Qdrant 로컬 인메모리 모드

> 이 노트북은 `src` 파일을 import하지 않고, 노트북 안의 코드만으로 실행되도록 구성한다.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

## 1. LLM 모델 생성

In [ ]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

## 2. 가장 단순한 모델 호출

In [ ]:
response = llm.invoke("RAG를 초보자도 이해할 수 있게 한 문장으로 설명해줘.")
print(response.content)

## 3. Prompt Template 사용

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 초보자를 돕는 AI 강사다. 답변은 간결하고 명확하게 작성한다."),
    ("human", "{question}")
])

chain = prompt | llm | StrOutputParser()

answer = chain.invoke({
    "question": "LangChain을 사용하는 이유를 설명해줘."
})

print(answer)

## 4. 출력 형식 지정

In [ ]:
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 AI 교육 강사다."),
    ("human", '''
다음 개념을 표 형식으로 정리해줘.

개념: {concept}

출력 형식:
| 항목 | 설명 |
|---|---|
''')
])

summary_chain = summary_prompt | llm | StrOutputParser()

print(summary_chain.invoke({"concept": "RAG"}))

## 핵심 정리

```python
prompt = ChatPromptTemplate.from_messages([...])
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
chain = prompt | llm | StrOutputParser()
result = chain.invoke({"question": "질문"})
```

- LangChain 1.x에서는 LCEL 방식으로 체인을 직관적으로 연결할 수 있다.
- RAG에서는 이 체인 앞에 문서 검색 단계가 추가된다.